In [3]:
from calendar import c
from numpy import argsort
import pandas as pd
import argparse
from bs4 import BeautifulSoup, Comment, MarkupResemblesLocatorWarning
import re
import urllib
import difflib
from nltk import sent_tokenize
import math
from tqdm import tqdm
import os
import warnings
from multiprocessing import Pool, cpu_count
import json
import weasyprint
tqdm.pandas()

from modular_preprocessing import *

In [ ]:
def calculate_color(score):
    # Assuming higher score means darker red
    intensity = 255 - int(score * 255)
    return f"rgb(255, {intensity}, {intensity})"

In [ ]:
def score_sentence(context_list):
    pass

- Parse HTML into sentences
- Look for sentence into context
- Store score
- Compute aggregated score
- Get color map
- Reinject colormap into html file

In [12]:
source_ID = "72930"  # Wikidata ID of the source entity
first_version = "8507794"  # First version of the source entity

# Path to the HTML file
file_path = f"/dlabdata1/tsoares/wikidumps/simplewiki-NS0-20231001/eval/pages/{source_ID}_{first_version}.html"

d = difflib.Differ()
file_1 = file_path
# read the first version
with open(file_1, 'r') as f:
    text_1 = f.read()

pdf1 = weasyprint.HTML(file_1).write_pdf()
open('one.pdf', 'wb').write(pdf1)

html_1 = BeautifulSoup(text_1, 'html.parser')
# the article is in the last div with class 'mw-parser-output'
html_1 = html_1.find_all('div', {'class': 'mw-parser-output'})
if len(html_1) == 0:
    print('No HTML found')
    print(f"{input['source_ID']}_{input['first_version']}.html")

html_1 = html_1[-1]
html_1_clean = simplify_html(html_1)

#save the simplified html
with open(f"{source_ID}_{first_version}_simplified.html", 'w') as f:
    f.write(str(html_1_clean))

pdf2 = weasyprint.HTML(f"{source_ID}_{first_version}_simplified.html").write_pdf()
open('two.pdf', 'wb').write(pdf2)


text_1 = "\n".join(
    [line for line in str(html_1_clean).split('\n') if line.strip() != ''])

sentences_1 = sent_tokenize(text_1)
sentences_1 = fix_sentence_tokenizer(sentences_1)


for sentence in sentences_1:
    soup = BeautifulSoup(sentence, 'html.parser')
    clean_text = soup.text.strip(' ')
    print(sentence)
    print(clean_text)
    print('\n\n\n')
                    

<div class="mw-content-ltr mw-parser-output" dir="ltr" lang="en">
<p><b>Quasars</b> or <b>quasi-stellar radio sources</b> are the most energetic and distant <a href="/wiki/Active_galactic_nucleus" title="Active galactic nucleus">active galactic nuclei</a> (AGN).

Quasars or quasi-stellar radio sources are the most energetic and distant active galactic nuclei (AGN).




</p><p>They are quite small in comparison with the energy they put out.
They are quite small in comparison with the energy they put out.




Quasars are not much larger than the <a href="/wiki/Solar_System" title="Solar System">Solar System</a>.
Quasars are not much larger than the Solar System.




The mechanism of brightness changes probably involves <a class="new" href="/w/index.php?title=Relativistic_beaming&amp;action=edit&amp;redlink=1" title="Relativistic beaming (not yet started)">relativistic beaming</a> of jets pointed nearly directly toward us.
The mechanism of brightness changes probably involves relativistic

In [63]:



# Sample overlapping subtexts and their scores
subtexts = ["The accretion discs of central supermassive black holes can convert about", "paragraph", "overlapping", "subtexts"]
scores = [0.3, 0.5, 0.9, 0.6]  # Assuming scores are normalized between 0 and 1

# Function to calculate color intensity based on score


# Read HTML content from file
with open(file_path, 'r', encoding='utf-8') as file:
    html_content = file.read()

# Parse HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Iterate through paragraphs and update with heat map colors
for paragraph in soup.find_all('p'):
    text = paragraph.get_text().lower()
    for subtext, score in zip(subtexts, scores):
        if subtext.lower() in text:
            color = calculate_color(score)
            paragraph['style'] = f"color: {color};"

# Save the updated HTML
updated_html = str(soup)
updated_file_path = "test_updated.html"
with open(updated_file_path, 'w', encoding='utf-8') as file:
    file.write(updated_html)
